# Step 1 — The model has no memory

*Step 1 of the AI in Industry lab*

---

## Read this before you run anything

You will call a language model directly and discover that it remembers nothing at all between calls — not even the message you sent five seconds ago.

**What you should end up understanding:** There is no memory on the server. Everything that feels like a conversation is a Python list your own code re-sends every time.

| | |
|---|---|
| **Cost** | 3 API calls |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes, every cell. Each run costs one API call. |

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b>Uses about 3 API calls.</b> Re-running cells is fine, it just uses a little more of your free quota each time.</div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

Everything that feels like memory is **a list your code re-sends**.

Run each cell, then read the note underneath it.

## First, the whole API

This is it. One function. Everything else in this lab is built on it.

In [ ]:
from labcore import chat

# chat() takes a list of messages and returns the reply text. That is all.
reply = chat([{"role": "user", "content": "In one sentence, what is a university?"}])
print(reply)

## Now tell it something to remember

In [ ]:
print(chat([{"role": "user", "content": "My roll number is 24BCE0142. Remember it."}]))

It will probably say something like *"I've noted that for the rest of our conversation."*

**It is not telling the truth.** Watch:

In [ ]:
# A brand new call. Nothing links it to the one above.
print(chat([{"role": "user", "content": "What is my roll number?"}]))

It has no idea.

There is no session, no memory, no profile on the server. Each call is completely independent.

## So how do chatbots remember?

**Your code re-sends the whole conversation every time.** Here it is, done by hand:

In [ ]:
history = [
    {"role": "user",      "content": "My roll number is 24BCE0142. Remember it."},
    {"role": "assistant", "content": "Noted, your roll number is 24BCE0142."},
    {"role": "user",      "content": "What is my roll number?"},
]

print(chat(history))

Now it knows — because we put the answer in the question.

That list is the entire mechanism of "conversation". **Memory is something you implement.**

---

## Now change it yourself

Edit the cell below and run it. Try:

- **Delete the middle message** (the `assistant` one). Does it still work?
- **Change the roll number** in the first message only. Which one does it report?
- **Add two more turns** and ask about something from the first one.

In [ ]:
history = [
    {"role": "user",      "content": "My roll number is 24BCE0142. Remember it."},
    {"role": "assistant", "content": "Noted, your roll number is 24BCE0142."},
    {"role": "user",      "content": "What is my roll number?"},
]

print(chat(history))

# Every message costs money and context space. This is why long chats get
# expensive: you are re-sending the entire history on every single turn.
print(f"\nSent {len(history)} messages, "
      f"{sum(len(m['content']) for m in history)} characters")

---

### Done with step 1

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.